# SmolLM3-3B LoRA — merken Phase 2 training (Colab A100)

Trains a LoRA adapter on top of `HuggingFaceTB/SmolLM3-3B` (Apache 2.0, text-only, standard `AutoModelForCausalLM`) to distill the winning Cerebras llama-8b Builder behavior into a local model, with an alignment twist: `contradicts` cases are replaced with an honest hedge (Jay's principle: *a memory system may doubt when it lacks data but must not state false facts as if true*).

**Pre-requisite:** Runtime → Change runtime type → A100 GPU → Save. Everything else is cloned from GitHub automatically.

**Expected wall time:** ~60-75 min on A100 for 414 rows × 3 epochs. Final output: `smollm3_adapter.zip` auto-downloaded.

## 1. Verify GPU

In [ ]:
!nvidia-smi | head -15

## 2. Clone the merken repo (Phase 2 branch)

Pulls the training script + training data from the `feature/phase2-ministral-lora` branch.

In [ ]:
!rm -rf /content/engram && git clone --depth 1 --branch feature/phase2-ministral-lora https://github.com/stffns/merken.git /content/engram 2>&1 | tail -5
!ls -lh /content/engram/experiments/phase2_training/
!ls -lh /content/engram/experiments/phase2_training/data/

## 3. Install deps

In [ ]:
!pip install --quiet --upgrade pip
# SmolLM3 needs transformers>=4.53 (stable PyPI release). Standard install.
!pip install --quiet --upgrade 'transformers>=4.53' 'peft>=0.14' 'trl>=0.15' 'datasets' 'accelerate' 'bitsandbytes' 'sentencepiece'
!pip list | grep -iE 'torch|transformers|peft|trl|datasets|accelerate' | head -10

## 4. Quick data sanity check

In [ ]:
import json, pathlib
DATA = pathlib.Path('/content/engram/experiments/phase2_training/data/train.jsonl')
rows = [json.loads(l) for l in DATA.open()]
print(f'{len(rows)} training rows')
from collections import Counter
c = Counter(r['teacher_verdict'] for r in rows)
print(f'verdict distribution: {dict(c)}')
print(f'hedge targets (teacher contradicts → forced "not enough information"): {c["contradicts"]}')
print('\n--- sample supports row ---')
sup = next(r for r in rows if r['teacher_verdict'] == 'supports')
print(f'qid={sup["qid"]}  target[:200]={sup["target"][:200]!r}')
print('\n--- sample contradicts row (target = hedge) ---')
con = next(r for r in rows if r['teacher_verdict'] == 'contradicts')
print(f'qid={con["qid"]}  target={con["target"]!r}')

## 5. Run training

Hyperparameters (script defaults):
- rank=16, alpha=32, dropout=0.05
- lr=1e-4, epochs=3, batch_size=8, grad_accum=2
- max_seq_len=4096
- bf16 (A100 supports it natively)

Loss should drop from ~2.0 to ~0.5-0.8 over 3 epochs.

In [ ]:
!cd /content/engram && python3 experiments/phase2_training/train_ministral_lora.py \
    --data experiments/phase2_training/data/train.jsonl \
    --output-dir /content/output 2>&1 | tee /content/train.log

## 6. Training metadata

In [ ]:
import json
meta = json.load(open('/content/output/adapter_final/training_metadata.json'))
print(json.dumps(meta, indent=2))

## 7. Bundle + auto-download adapter

Adapter is ~40MB (LoRA weights only, no base model merged in). Downloaded as a zip.

In [ ]:
import shutil
shutil.make_archive('/content/smollm3_adapter', 'zip', '/content/output/adapter_final')
!ls -lh /content/smollm3_adapter.zip
from google.colab import files
files.download('/content/smollm3_adapter.zip')
files.download('/content/train.log')

## Local next steps

```
cd ~/Desktop/Personal/Projects/engram
git checkout feature/phase2-ministral-lora
mkdir -p experiments/phase2_training/adapter
unzip ~/Downloads/smollm3_adapter.zip -d experiments/phase2_training/adapter/
```

Then conversion to MLX + gate eval runs locally on M-series via mlx_lm.fuse + pipeline_runner.